# 📊 02_02 — Filtering & GroupBy: Deeper and Wider
### Data Analyst Curriculum | Pandas

---

### What you'll learn
1. Filtering with `.isin()` and `~` (NOT)
2. Multi-column `.groupby()`
3. `.agg()` — multiple stats at once
4. Pivot tables with `.pivot_table()`
5. `.transform()` — adding group stats back to the original DataFrame

### The dataset
A small gym — 12 members. Small enough to see every row,
with enough variety to make groupby interesting.

---

## 🔧 Setup

In [2]:
import pandas as pd

data = {
    'name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank',
                   'Grace', 'Hank', 'Iris', 'Jack', 'Karen', 'Leo'],
    'plan':       ['basic', 'premium', 'basic', 'premium', 'basic', 'premium',
                   'basic', 'premium', 'elite', 'elite', 'basic', 'elite'],
    'location':   ['downtown', 'uptown', 'downtown', 'suburbs', 'suburbs', 'uptown',
                   'downtown', 'suburbs', 'uptown', 'downtown', 'uptown', 'suburbs'],
    'visits':     [4, 12, 7, 20, 3, 15, 8, 18, 25, 22, 5, 30],
    'monthly_fee':[30, 60, 30, 60, 30, 60, 30, 60, 90, 90, 30, 90],
    'age':        [28, 34, 45, 29, 52, 41, 36, 27, 31, 48, 55, 23],
}

df = pd.DataFrame(data)
# df # commented for easy phone navigation

## 🔎 Section 1: Smarter Filtering with `.isin()` and `~`

You already know how to filter with `==` for one value and `|` for multiple.
There's a cleaner way to filter for multiple values: `.isin()`.

### 1a. `.isin()`

Instead of:
```python
df[(df['plan'] == 'basic') | (df['plan'] == 'premium')]
```

You can write:
```python
df[df['plan'].isin(['basic', 'premium'])]
```

Same result, much cleaner when you have 3+ values to match.

In [3]:
# Members on basic or premium plans
df[df['plan'].isin(['basic', 'premium'])]

,name,plan,location,visits,monthly_fee,age
0,Alice,basic,downtown,4,30,28
1,Bob,premium,uptown,12,60,34
2,Carol,basic,downtown,7,30,45
3,Dave,premium,suburbs,20,60,29
4,Eve,basic,suburbs,3,30,52
5,Frank,premium,uptown,15,60,41
6,Grace,basic,downtown,8,30,36
7,Hank,premium,suburbs,18,60,27
10,Karen,basic,uptown,5,30,55


In [4]:
# Members located downtown or uptown
df[df['location'].isin(['downtown', 'uptown'])]

,name,plan,location,visits,monthly_fee,age
0,Alice,basic,downtown,4,30,28
1,Bob,premium,uptown,12,60,34
2,Carol,basic,downtown,7,30,45
5,Frank,premium,uptown,15,60,41
6,Grace,basic,downtown,8,30,36
8,Iris,elite,uptown,25,90,31
9,Jack,elite,downtown,22,90,48
10,Karen,basic,uptown,5,30,55


### 1b. `~` — the NOT operator

`~` flips a True/False condition to its opposite.
It goes right before the condition, inside the outer brackets.

```python
df[~df['plan'].isin(['basic', 'premium'])]  # everyone NOT on basic or premium
df[~(df['visits'] > 10)]                    # everyone with 10 or fewer visits
```

In [5]:
# Members NOT on basic or premium (i.e. elite only)
df[~df['plan'].isin(['basic', 'premium'])]

,name,plan,location,visits,monthly_fee,age
8,Iris,elite,uptown,25,90,31
9,Jack,elite,downtown,22,90,48
11,Leo,elite,suburbs,30,90,23


In [6]:
# Members who do NOT visit more than 10 times
# Same as df[df['visits'] <= 10] but useful when the condition is complex
df[~(df['visits'] > 10)]

,name,plan,location,visits,monthly_fee,age
0,Alice,basic,downtown,4,30,28
2,Carol,basic,downtown,7,30,45
4,Eve,basic,suburbs,3,30,52
6,Grace,basic,downtown,8,30,36
10,Karen,basic,uptown,5,30,55


### 🏋️ Exercise 1a
Filter `df` to members located in `'suburbs'` or `'uptown'` using `.isin()`.
Print how many matched.

```python
# Your code here
```

In [7]:
# Your code here

subs_or_up = df[df['location'].isin(['suburbs','uptown'])]

print(f" the amount of gym attendants in the suburbs or uptown locations are {len(subs_or_up)}")






 the amount of gym attendants in the suburbs or uptown locations are 8


### 🏋️ Exercise 1b
Filter `df` to members who are NOT on the `'basic'` plan using `~`.
Print how many matched.

```python
# Your code here
```

In [8]:
# Your code here
non_basic = df[~df['plan'].isin(['basic'])]

print(f"The number of people not on a basic plan are {len(non_basic)}")






The number of people not on a basic plan are 7


### 🏋️ Exercise 1c
Filter `df` to members who are in `'downtown'` or `'suburbs'` AND have more than 10 visits.
Print how many matched and their average monthly fee.

```python
# Your code here
```

In [9]:
# Your code here
ex_onec = df[(df['location'].isin(['downtown','suburbs'])) & (df['visits'] > 10)]
print(f"There are {len(ex_onec)} members who go Downtown or the Suburbs and have more than 10 visits")







There are 4 members who go Downtown or the Suburbs and have more than 10 visits


## 📊 Section 2: Grouping by Multiple Columns

So far you've grouped by one column at a time.
You can pass a list to `.groupby()` to group by two columns at once —
every unique *combination* gets its own group.

```python
df.groupby(['col1', 'col2'])['value'].aggregation()
```

The result has a **multi-level index** — one level per group column.
Use `.reset_index()` at the end to flatten it back into a normal DataFrame.

In [10]:
# Average visits per plan AND location combination
df.groupby(['plan', 'location'])['visits'].mean()
# The value is where the aggregation gets applied

plan     location
basic    downtown     6.333333
         suburbs      3.000000
         uptown       5.000000
elite    downtown    22.000000
         suburbs     30.000000
         uptown      25.000000
premium  suburbs     19.000000
         uptown      13.500000
Name: visits, dtype: float64

In [11]:
# Same but flattened with reset_index() — easier to read and work with
df.groupby(['plan', 'location'])['visits'].mean().reset_index()
# multiple rows

,plan,location,visits
0,basic,downtown,6.333333
1,basic,suburbs,3.000000
2,basic,uptown,5.000000
3,elite,downtown,22.000000
4,elite,suburbs,30.000000
5,elite,uptown,25.000000
6,premium,suburbs,19.000000
7,premium,uptown,13.500000


In [12]:
# Total monthly fee revenue per plan and location
df.groupby(['plan', 'location'])['monthly_fee'].sum().reset_index()

,plan,location,monthly_fee
0,basic,downtown,90
1,basic,suburbs,30
2,basic,uptown,30
3,elite,downtown,90
4,elite,suburbs,90
5,elite,uptown,90
6,premium,suburbs,120
7,premium,uptown,120


### 🏋️ Exercise 2a
Find the average age per plan type.
Print which plan has the oldest members on average.

```python
# Your code here
```

In [13]:
# Your code here
avg_age_per_plan = df.groupby('plan')['age'].mean()
print(f"The plan with the oldest members on average is {avg_age_per_plan.idxmax()}")





The plan with the oldest members on average is basic


### 🏋️ Exercise 2b
Find the total visits per plan AND location combination using `.reset_index()`.
Display the result as a DataFrame.

```python
# Your code here
```

In [14]:
# Your code here
df.groupby(['plan','location'])['visits'].sum().reset_index()

,plan,location,visits
0,basic,downtown,19
1,basic,suburbs,3
2,basic,uptown,5
3,elite,downtown,22
4,elite,suburbs,30
5,elite,uptown,25
6,premium,suburbs,38
7,premium,uptown,27


### 🏋️ Exercise 2c
Find the average monthly fee per location.
Then find the average visits per location.
Print a sentence for each location comparing fee vs visits.

```python
# Your code here
```

In [15]:
# Your code here
avg_fee_per_loc = df.groupby('location')['monthly_fee'].mean()
avg_visits_per_loc = df.groupby('location')['visits'].mean()

for location in avg_fee_per_loc.index:
  print(f"{location}: avg fee is ${avg_fee_per_loc[location]:,.2f} and avg visits are {avg_visits_per_loc[location]:,.1f}")




downtown: avg fee is $45.00 and avg visits are 10.2
suburbs: avg fee is $60.00 and avg visits are 17.8
uptown: avg fee is $60.00 and avg visits are 14.2


## 🔢 Section 3: Multiple Stats at Once with `.agg()`

So far you've called one aggregation at a time — `.sum()`, `.mean()`, etc.
`.agg()` lets you run several at once and get them all in one table.

```python
df.groupby('col')['value'].agg(['mean', 'min', 'max', 'count'])
```

You can also rename the output columns using a dictionary:
```python
df.groupby('col')['value'].agg(
    average='mean',
    lowest='min',
    highest='max',
    total='count'
)
```

In [16]:
# Multiple stats on visits per plan
df.groupby('plan')['visits'].agg(['mean', 'min', 'max', 'count'])

,mean,min,max,count
plan,,,,
basic,5.400000,3,8,5
elite,25.666667,22,30,3
premium,16.250000,12,20,4


In [17]:
# Same thing with renamed columns — cleaner output
df.groupby('plan')['visits'].agg(
    avg_visits='mean',
    min_visits='min',
    max_visits='max',
    member_count='count'
)

,avg_visits,min_visits,max_visits,member_count
plan,,,,
basic,5.400000,3,8,5
elite,25.666667,22,30,3
premium,16.250000,12,20,4


In [18]:
# .agg() on multiple columns at once using a dictionary
df.groupby('plan').agg(
    avg_visits=('visits', 'mean'),
    total_revenue=('monthly_fee', 'sum'),
    member_count=('name', 'count')
)

,avg_visits,total_revenue,member_count
plan,,,
basic,5.400000,150,5
elite,25.666667,270,3
premium,16.250000,240,4


### 🏋️ Exercise 3a
Use `.agg()` to find the mean, min, and max of `monthly_fee` per `plan`.

```python
# Your code here
```

In [19]:
# Your code here
df.groupby('plan')['monthly_fee'].agg(['mean','min','max'])


,mean,min,max
plan,,,
basic,30.0,30,30
elite,90.0,90,90
premium,60.0,60,60


### 🏋️ Exercise 3b
Use `.agg()` with renamed columns to find:
- average visits per location (call it `avg_visits`)
- total monthly fee per location (call it `total_revenue`)

```python
# Your code here
```

In [20]:
# Your code here
df.groupby('location').agg(
    avg_visits=('visits','mean'),
    total_revenue=('monthly_fee','sum')
)

# print("the suburbs and uptown locations make the same total revenue, despite the suburbs having more visits on average")






the suburbs and uptown locations make the same total revenue, despite the suburbs having more visits on average


### 🏋️ Exercise 3c
Use `.agg()` across multiple columns to find per plan:
- average age (call it `avg_age`)
- average visits (call it `avg_visits`)
- total revenue (call it `total_revenue`)
- member count (call it `members`)

```python
# Your code here
```

In [36]:
# Your code here
df.groupby('plan').agg(
    avg_age=('age','mean'),
    avg_visits=('visits','mean'),
    total_revenue=('monthly_fee','sum'),
    members=('name','count')
)





,avg_age,avg_visits,total_revenue,members
plan,,,,
basic,43.20,5.400000,150,5
elite,34.00,25.666667,270,3
premium,32.75,16.250000,240,4


## 🔄 Section 4: Pivot Tables with `.pivot_table()`

A pivot table is a way to reshape grouped data into a grid —
one column becomes the rows, another becomes the columns, and the values fill the cells.

This is the same concept as pivot tables in Excel.

```python
df.pivot_table(values='value_col', index='row_col', columns='col_col', aggfunc='mean')
```

- `values` — the column you want to summarize
- `index` — what becomes the row labels
- `columns` — what becomes the column headers
- `aggfunc` — how to summarize (same options as groupby: `'mean'`, `'sum'`, etc.)

In [22]:
# Average visits — rows = location, columns = plan
df.pivot_table(values='visits', index='location', columns='plan', aggfunc='mean')

plan,basic,elite,premium
location,,,
downtown,6.333333,22.0,NaN
suburbs,3.000000,30.0,19.0
uptown,5.000000,25.0,13.5


In [23]:
# Total monthly fee — rows = plan, columns = location
df.pivot_table(values='monthly_fee', index='plan', columns='location', aggfunc='sum')

location,downtown,suburbs,uptown
plan,,,
basic,90.0,30.0,30.0
elite,90.0,90.0,90.0
premium,NaN,120.0,120.0


In [24]:
# Count of members — rows = location, columns = plan
df.pivot_table(values='name', index='location', columns='plan', aggfunc='count')

plan,basic,elite,premium
location,,,
downtown,3.0,1.0,NaN
suburbs,1.0,1.0,2.0
uptown,1.0,1.0,2.0


Notice the `NaN` values — those are combinations that don't exist in the data.
For example if no elite members are downtown, that cell is empty.

You can fill them with 0 using `fill_value=0`:
```python
df.pivot_table(..., fill_value=0)
```

In [25]:
# Same count table but with 0 instead of NaN
df.pivot_table(values='name', index='location', columns='plan', aggfunc='count', fill_value=0)

plan,basic,elite,premium
location,,,
downtown,3,1,0
suburbs,1,1,2
uptown,1,1,2


### 🏋️ Exercise 4a
Create a pivot table showing average `age` with `plan` as rows and `location` as columns.

```python
# Your code here
```

In [26]:
# Your code here

### 🏋️ Exercise 4b
Create a pivot table showing total `visits` with `location` as rows and `plan` as columns.
Fill any missing combinations with 0.

```python
# Your code here
```

In [27]:
# Your code here

### 🏋️ Exercise 4c
Create a pivot table showing the count of members per `plan` and `location`.
Fill missing with 0. Which combination has the most members?

```python
# Your code here
```

In [28]:
# Your code here

## ✨ Section 5: Adding Group Stats Back with `.transform()`

`.groupby().agg()` collapses your DataFrame into a summary.
But sometimes you want to *keep* all the original rows and just
add a new column showing each row's group stat alongside it.

That's what `.transform()` does — it returns a value for every row,
matching each row to its group's stat.

```python
df['new_col'] = df.groupby('group_col')['value_col'].transform('mean')
```

Every row gets the average of its own group — not the overall average.

In [29]:
# Add a column showing the average visits for each member's plan
df['plan_avg_visits'] = df.groupby('plan')['visits'].transform('mean')
df[['name', 'plan', 'visits', 'plan_avg_visits']]

,name,plan,visits,plan_avg_visits
0,Alice,basic,4,5.400000
1,Bob,premium,12,16.250000
2,Carol,basic,7,5.400000
3,Dave,premium,20,16.250000
4,Eve,basic,3,5.400000
5,Frank,premium,15,16.250000
6,Grace,basic,8,5.400000
7,Hank,premium,18,16.250000
8,Iris,elite,25,25.666667
9,Jack,elite,22,25.666667


In [30]:
# Now you can compare each member to their plan's average
df['visits_vs_plan_avg'] = df['visits'] - df['plan_avg_visits']
df[['name', 'plan', 'visits', 'plan_avg_visits', 'visits_vs_plan_avg']]

,name,plan,visits,plan_avg_visits,visits_vs_plan_avg
0,Alice,basic,4,5.400000,-1.400000
1,Bob,premium,12,16.250000,-4.250000
2,Carol,basic,7,5.400000,1.600000
3,Dave,premium,20,16.250000,3.750000
4,Eve,basic,3,5.400000,-2.400000
5,Frank,premium,15,16.250000,-1.250000
6,Grace,basic,8,5.400000,2.600000
7,Hank,premium,18,16.250000,1.750000
8,Iris,elite,25,25.666667,-0.666667
9,Jack,elite,22,25.666667,-3.666667


This is powerful for spotting outliers within a group —
who visits way more or less than others on the same plan?

### 🏋️ Exercise 5a
Add a column called `location_avg_fee` showing the average monthly fee
for each member's location. Display `name`, `location`, `monthly_fee`, and `location_avg_fee`.

```python
# Your code here
```

In [31]:
# Your code here

### 🏋️ Exercise 5b
Using the column from 5a, add another column called `fee_vs_location_avg`
showing how much each member's fee differs from their location's average.
Who has the biggest positive difference?

```python
# Your code here
```

In [32]:
# Your code here

### 🏋️ Exercise 5c
Add a column called `plan_total_visits` showing the total visits
for all members on each member's plan. Then add a column called `visit_share`
showing what percentage of their plan's total visits each member accounts for.

```python
# Your code here
```

In [33]:
# Your code here

## 🏁 Capstone: Gym Member Analysis

Answer these questions using any combination of what you've learned.
No method hints — read the question and decide what to reach for.

1. Which location has the highest average visits per member?
2. Among premium and elite members only, which plan generates more total revenue?
3. Build a pivot table showing average visits by plan and location.
   Which plan-location combination has the highest engagement?
4. Add a column showing how each member's visits compare to their location's average.
   Who are the top 3 most active members relative to their location?
5. Write 3–4 print statements presenting your findings as a report to the gym owner.

```python
# Your code here
```

In [34]:
# Your code here

---
## ✅ Done with 02_02

New tools in your kit:
- `.isin()` and `~` for cleaner filtering
- Multi-column `.groupby()`
- `.agg()` for multiple stats at once
- `.pivot_table()` for grid-style summaries
- `.transform()` for adding group stats back to the original DataFrame

Next up: **02_03 — Cleaning and Feature Engineering**